# SHACL UI — User Guide

SHACL 1.2 User Interfaces (`shui:`, namespace `http://www.w3.org/ns/shacl-ui#`) defines a widget/editor/viewer vocabulary for generating data-entry forms from shapes. It's a separate vocabulary from `sh:`, and its own spec is scoped entirely to form rendering — no new constraint types, no new validation semantics, nothing a validation or rules engine is required to implement.

What starshacl actually does with `shui:`: nothing at validation time, by design — annotations pass through `validate()`/`apply_rules()`/the meta-shacl preflight as inert, unrecognized triples, the same way `sh:order`/`sh:group` already do. This guide is short because that's genuinely the whole runtime story; see the end for what's deliberately not built.

## How to run this notebook

1. `pip install "git+https://github.com/hidden-graph/starlayer.git"` (or install the three packages editable from a local checkout — see the root [README](../../README.md)).
2. Run cells top to bottom.

In [1]:
from starlayergraph import StarLayerGraph, Namespace
from starshacl import StarShaclValidator

EX = Namespace("http://example.org/")

## `shui:` annotations pass through unchanged

`shui:editor`/`shui:viewer` pick a form widget for a property; `shui:propertyRole` marks a property with a special UI role (`shui:LabelRole` — "use this property's value as the resource's display label" — is the one built-in role). None of this affects whether the shape validates.

In [2]:
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ; ex:birthDate "1990-01-01"^^<http://www.w3.org/2001/XMLSchema#date> .
    ex:bob a ex:Person .
""", format="turtle")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix shui: <http://www.w3.org/ns/shacl-ui#> .
    ex:PersonShape a sh:NodeShape ;
      sh:targetClass ex:Person ;
      sh:property [
        sh:path ex:birthDate ;
        sh:minCount 1 ;
        sh:datatype <http://www.w3.org/2001/XMLSchema#date> ;
        sh:name "Birth Date" ;
        sh:order 1 ;
        shui:editor shui:DatePickerEditor ;
        shui:viewer shui:LabelViewer ;
        shui:propertyRole shui:LabelRole ;
      ] .
""", format="turtle")

# validate(): the shui: annotations don't change which nodes conform
result = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes, meta_shacl=False)
print("conforms:", result.conforms)
print("ex:bob flagged (missing birthDate):", "ex:bob" in result.report_text)
print("ex:alice flagged:", "ex:alice" in result.report_text.split("Focus Node:")[-1])

# meta-shacl: the shui: predicates are simply unrecognized by the meta-shapes, not rejected -
# a conforming data graph here isolates that question from ex:bob's violation above
conforming_data = StarLayerGraph()
conforming_data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ; ex:birthDate "1990-01-01"^^<http://www.w3.org/2001/XMLSchema#date> .
""", format="turtle")
meta_result = StarShaclValidator().validate(data_graph=conforming_data, shacl_graph=shapes, meta_shacl=True)
print("meta_shacl conforms:", meta_result.conforms)

conforms: False
ex:bob flagged (missing birthDate): True
ex:alice flagged: False
meta_shacl conforms: True


In [3]:
# apply_rules(): a shui:editor annotation on the same property a rule reads from doesn't interfere
data2 = StarLayerGraph()
data2.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice ex:parent ex:carol .
""", format="turtle")

shapes2 = StarLayerGraph()
shapes2.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix shui: <http://www.w3.org/ns/shacl-ui#> .
    ex:R a sh:NodeShape ;
      sh:targetSubjectsOf ex:parent ;
      sh:property [ sh:path ex:parent ; shui:editor shui:InstancesSelectEditor ] ;
      sh:rule [
        a sh:TripleRule ;
        sh:subject sh:this ; sh:predicate ex:hasParent ; sh:object [ sh:path ex:parent ] ;
      ] .
""", format="turtle")

result2 = StarShaclValidator().apply_rules(data_graph=data2, shacl_graph=shapes2)
print("rule fired despite the shui: annotation:", (EX.alice, EX.hasParent, EX.carol) in result2.data_graph)

rule fired despite the shui: annotation: True


## Built-in widget vocabulary

Around 26 built-in editor/viewer instances exist (`shui:TextFieldEditor`, `shui:NumberFieldEditor`, `shui:DatePickerEditor`, `shui:EnumSelectEditor`, `shui:InstancesSelectEditor`, `shui:HyperlinkViewer`, `shui:ImageViewer`, and others) — reference documentation only, not something starshacl interprets. Full list: `packages/shacl/docs/shacl-presentation-content.md`'s "SHACL 1.2 UI Widgets" section.

Separately, starshacl has its own simpler, unrelated mechanism for suggesting a widget type per predicate — `stsh:widgetType`, a plain string tag (`text`, `multiline`, `int`, `bool`, `datatype`, `in_list`, ...) documented for every shape-authoring predicate in this project's own presentation-content vocabulary. It's a default, not a binding UI contract, and it's a separate concern from `shui:editor`/`shui:viewer` — the two aren't wired together.

## Further work

- **The widget-*selection* algorithm** (`shui:WidgetScore`/`shui:WidgetAcceptMatcher` — given a shape, a focus node, and a scoring graph, deterministically pick the best-matching `shui:editor`/`shui:viewer`) is a genuinely separate, optional feature and is not built. Compatibility of the annotations themselves (this guide) is confirmed; the algorithm that would actually *use* them to pick a widget at runtime is not.